# 从数据集获取嵌入

本 notebook 演示如何从大型数据集中获取嵌入（embeddings）。


## 1. 加载数据集

本示例使用的数据集是来自 Amazon 的 [fine-food reviews](https://www.kaggle.com/snap/amazon-fine-food-reviews)。该数据集共包含 568,454 条截至 2012 年 10 月 Amazon 用户留下的食品评论。为便于演示，我们将使用其中的一个子集，即最近的 1,000 条评论。这些评论为英文，倾向于正面或负面。每条评论都包含 ProductId、UserId、Score、评论标题（Summary）和评论正文（Text）。

我们会将评论标题和评论正文合并为一段组合文本。模型将对这段组合文本进行编码，并输出一个向量嵌入。

要运行本 notebook，你需要安装：pandas、openai、transformers、plotly、matplotlib、scikit-learn、torch（transformer 的依赖）、torchvision 和 scipy。

In [20]:
import pandas as pd
import tiktoken

from utils.embeddings_utils import get_embedding

In [16]:
embedding_model = "text-embedding-3-small"
embedding_encoding = "cl100k_base"
max_tokens = 8000  # the maximum for text-embedding-3-small is 8191

In [17]:
# load & inspect dataset
input_datapath = "data/fine_food_reviews_1k.csv"  # to save space, we provide a pre-filtered dataset
df = pd.read_csv(input_datapath, index_col=0)
df = df[["Time", "ProductId", "UserId", "Score", "Summary", "Text"]]
df = df.dropna()
df["combined"] = (
    "Title: " + df.Summary.str.strip() + "; Content: " + df.Text.str.strip()
)
df.head(2)

,Time,ProductId,UserId,Score,Summary,Text,combined
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...


In [18]:
# subsample to 1k most recent reviews and remove samples that are too long
top_n = 1000
df = df.sort_values("Time").tail(top_n * 2)  # first cut to first 2k entries, assuming less than half will be filtered out
df.drop("Time", axis=1, inplace=True)

encoding = tiktoken.get_encoding(embedding_encoding)

# omit reviews that are too long to embed
df["n_tokens"] = df.combined.apply(lambda x: len(encoding.encode(x)))
df = df[df.n_tokens <= max_tokens].tail(top_n)
len(df)

1000

## 2. 获取嵌入并保存以便后续复用

In [21]:
# Ensure you have your API key set in your environment per the README: https://github.com/openai/openai-python#usage

# This may take a few minutes
df["embedding"] = df.combined.apply(lambda x: get_embedding(x, model=embedding_model))
df.to_csv("data/fine_food_reviews_with_embeddings_1k.csv")

In [22]:
a = get_embedding("hi", model=embedding_model)